## Import du dataframe

In [ ]:
import pandas as pd
import numpy as np

# Bibliothèques pour la visualisation
import matplotlib.pyplot as plt
import seaborn as sns

df_household = pd.read_csv('../data/src/household.csv', sep=';', dtype={'GEO': str})
df_population = pd.read_csv('../data/src/population.csv', sep=';', dtype={'GEO': str})
df_conjugality = pd.read_csv('../data/src/conjugality.csv', sep=';', dtype={'GEO': str})
df_election =  pd.read_csv('../data/src/election.csv', sep=';', dtype={'Code_INSEE': str})

display(df_household)
display(df_election)
display(df_population)

In [ ]:
#df_household[df_household["GEO"] == "56008"]

In [ ]:
df_household["TPH"].unique()
df_population["AGE"].unique()
df_population["SOCIO_PROF_CAT"].unique()

In [ ]:
df_household_line = (
    df_household.pivot_table(
        index=["GEO", "CITY"],
        columns="TPH",
        values="OBS_VALUE",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

display(df_household_line)

In [ ]:
df = df_household_line.merge(
    df_election,
    left_on="GEO",
    right_on="Code_INSEE",
    how="inner"
)

display(df)

In [ ]:
df["Votants"].sum().item()

In [ ]:
df_election["Votants"].sum().item()

In [ ]:
df.columns

In [ ]:
df =df.drop('Code_INSEE', axis=1)
df =df.drop('Libellé de la commune', axis=1)

In [ ]:
display(df)

In [ ]:
df.info()

In [ ]:
# df_gauche = df.pop('pct_gauche')
# df_centre = df.pop('pct_centre')
# df_droite = df.pop('pct_droite')

In [ ]:
top_5 = df.sort_values(by="Votants", ascending=False).head(5)

top_5[["CITY", "Votants"]]

In [ ]:
df['Homme_seul_en_pct'] = (df['Homme seul'] / df['Total']) * 100
df['Femme_seul_en_pct'] = (df['Femme seule'] / df['Total']) * 100
df['Fpcae_en_pct'] = (df['Famille principale couple avec enfants'] / df['Total']) * 100
df['Fpcse_en_pct'] = (df['Famille principale couple sans enfants'] / df['Total']) * 100
df['Fpm_en_pct'] = (df['Famille principale monoparentale'] / df['Total']) * 100 
df['Mcuf_en_pct'] = (df['Ménage comprenant une famille'] / df['Total']) * 100 #1 147h
df['Msfapp_en_pct'] = (df['Ménage sans famille à plusieurs personnes '] / df['Total']) * 100
df['Maup_en_pct'] = (df['Ménage à une personne'] / df['Total']) * 100

display(df)

In [ ]:
filtered = df[df["Total"] > 100]

display(filtered)

In [ ]:
data = filtered.copy()

x = ["Homme_seul_en_pct", 'Femme_seul_en_pct' ,'Fpcae_en_pct', "Fpcse_en_pct", 'Fpm_en_pct','Mcuf_en_pct', 'Msfapp_en_pct', 'Maup_en_pct']
y = ["pct_gauche", "pct_centre", "pct_droite"]

corr = data[x + y].corr()

res = corr.loc[x, y]

res

In [ ]:
def commune(habitants):
    if habitants < 2000:
        return "rurale"
    elif habitants < 10000:
        return "petite_commune"
    elif habitants < 50000:
        return "ville_moyenne"
    elif habitants < 100000:
        return "grande_ville"
    else:
        return "tres_grande_ville"

filtered["categorie_commune"] = filtered["Total"].apply(commune)
filtered[["CITY", "Total", "categorie_commune"]].head()

In [ ]:
filtered.groupby("categorie_commune")[["pct_gauche", "pct_centre", "pct_droite", "pct_blancs"]].mean()